## U24AI074

### Assignment 2

You are given a dataset in JSON format text_segmentation_dataset.json. The dataset is a
snapshot from the large Brown corpus. You need to implement two text segmentation
techniques to segment the text into words and report their performance.
1. Greedy Based Approach that matches the longest word
2. Dynamic Programming Approach that increases the log probability of the text [Hint:
Frequencies of Words are given]

You need to report two evaluation metrics.
1. Accuracy
2. Edit Distance

In [2]:
import re
import pandas as pd
import json
import math


In [3]:
with open("text_segmentation_dataset.json", "r") as f:
    data=json.load(f)

word_counts=data["word_counts"]
test_cases=data["test_cases"]
total_words=data["metadata"]["total_corpus_words"]

vocabulary=set(word_counts.keys())

MAX_WORD_LENGTH=max(len(word) for word in vocabulary )

### Greedy

In [4]:
def greedy_segment(text):
    result=[]
    i=0
    while i<len(text):
        found_word=None
        max_len=min(MAX_WORD_LENGTH,len(text)-i)

        for length in range(max_len,0,-1):
            word=text[i:i+length]
            if word in vocabulary:
                found_word=word
                break

        if found_word is None:
            found_word=text[i]

        result.append(found_word)
        i+=len(found_word)


    return result


#### memoized

In [5]:
def f(i,text,word_counts,total_words,vocabulary,dp):
    if i==len(text):
        return 0
    best=float("-inf")
    if dp[i] is not None:
        return dp[i]
    for j in range(i+1, len(text)+1, 1):
        word=text[i:j]
        if word in vocabulary:
            probability = word_counts[word]/total_words
            score=math.log(probability) + f(j,text,word_counts,total_words,vocabulary,dp)
            best=max(best,score)
        
    dp[i]=best
    return dp[i]




In [6]:
text=data["test_cases"][0]["input"]
dp=[None]*(len(text)+1)
print(f(0,text,word_counts,total_words,vocabulary,dp))

-50.4370051198282


#### Tabulation

In [7]:
def dp_tab(text,word_counts,total_words,vocabulary):
    n=len(text)
    dp=[float("-inf")]*(n+1)
    previous=[None]*(n+1)
    dp[0]=0
    for i in range(n+1):
        for j in range (0,i):
            word= text[j:i]
            if word in vocabulary:
                probability=word_counts[word]/total_words
                score=dp[j]+math.log(probability)
                if score>dp[i]:
                    dp[i]=score
                    previous[i]=j

    words=[]
    position=n

    while position>0:
        j=previous[position]
        if j is None:
            return None
        words.append(text[j:position])
        position=j

    words.reverse()
    return words


In [8]:
print(greedy_segment(text))
print(dp_tab(text,word_counts,total_words,vocabulary))

['it', 'that', 'the', 'city', 'takes', 't', 'e', 'p', 's', 'to', 'this', 'problem']
['it', 'that', 'the', 'city', 'take', 'steps', 'to', 'this', 'problem']


#### Edit Distance

In [ ]:
def edit_distance(a,b):
    m=len(a)
    n=len(b)
    dp=[[0]*(n+1) for _ in range (0,m+1)]
    for i in range(m+1):
        dp[i][0]=i

    for j in range(n+1):
            dp[0][j]=j
    for i in range(1,m+1):
        for j in range(1,n+1):
            if a[i-1]==b[j-1]:
                cost=0
            else:
                 cost=1

            dp[i][j]=min(
                 1+dp[i-1][j], #deletion
                 1+dp[i][j-1], #insertion
                 cost+dp[i-1][j-1] #replace
            )

    return dp[m][n]


#### Evaluation

In [10]:
greedy_correct=0
dp_correct=0

greedy_edit_total=0
dp_edit_total=0

for case in test_cases:
    test=case["input"]
    ground_truth=case["ground_truth"].split()

    greedy_result=greedy_segment(test)
    dp_result=dp_tab(test,word_counts,total_words,vocabulary)


    if greedy_result==ground_truth:
        greedy_correct+=1

    if dp_result==ground_truth:
        dp_correct+=1

    greedy_edit_total+=edit_distance(greedy_result,ground_truth)
    dp_edit_total+=edit_distance(dp_result, ground_truth)

    

#### Final Results

In [11]:
number_of_cases=len(test_cases)
greedy_accuracy=(greedy_correct/number_of_cases)*100
dp_accuracy=(dp_correct/number_of_cases)*100
greedy_avg_edit=greedy_edit_total/number_of_cases
dp_avg_edit=dp_edit_total/number_of_cases


print("Greedy Segmentation")
print("Correct cases:", greedy_correct)
print("Accuracy:", greedy_accuracy, "%")
print("Average Edit Distance:", greedy_avg_edit)

print("DP segmentation")
print("Correct cases:", dp_correct)
print("Accuracy:", dp_accuracy, "%")
print("Average Edit Distance:", dp_avg_edit)

Greedy Segmentation
Correct cases: 691
Accuracy: 69.1 %
Average Edit Distance: 1.26
DP segmentation
Correct cases: 982
Accuracy: 98.2 %
Average Edit Distance: 0.037
